# T1-12 Same held-out cases: base vs floor vs adapter

**Time:** 10:45–12:00 &nbsp;|&nbsp; **Evidence:** confusion matrix + harmful-error count in `logs/`

**Notebook learning outcomes**

- Re-score **criteria** on the frozen set with the **same** one-criterion prompt.
- Put the **rule floor** next to the base (if evidence is not a substring, force `human_review`).
- Read the **redaction** 40-document comparison from run files (HOLD / PASS CHECKS). Nothing here promises the adapter is better.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


**Data ban:** CivicDesk fiction only. The frozen held-out cases are a **test** set. Do not train on them. Do not paste ministry CVs, tickets or minutes.


## Overnight readers

The **base** is OLMo as served on Day 1. Frozen.

A **LoRA** would be a small file on that base. This pack may not contain measured adapter weights yet. If `olmo_adapter_v1.json` is missing, the redaction comparison uses a **demo** file labelled not scored, and criteria comparison uses the **floor**, not a fake win.


## Ministry link

Labour: `human_review` recall and the count of **human_review → does_not_meet** (an applicant marked as failing for something the CV never addressed).


## 0. Backend


In [ ]:
# [CHANGE] olmo is the classroom default. phi needs :8001. demo = labelled stand-ins, not scored.
BACKEND = "olmo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Adapter card


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.print_adapter_card()


## 2. Criteria — this session's base numbers

If pre-flight did not run live, this cell says so. Do not type percentages from memory.


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.print_preflight_metrics()


## 3. Simplest thing first: the evidence floor

If the model's evidence string is not copied from the résumé, force `human_review`. If this floor gets most of the gain, that is an honest result.


In [ ]:
# [RUN]
import civicdesk_ft as ft

floored = ft.floor_from_preflight()


## 4. Redaction — the same 40 documents as Day 1

Per document: **HOLD** or **PASS CHECKS**. Per run: **meets** or **does not meet** the example rule. A one- or two-document swing is noise.


In [ ]:
# [RUN]
import civicdesk_ft as ft

cmp = ft.compare_runs(ft.baseline_run_path(), ft.adapter_run_path())
ft.save_log("T1-12_compare", cmp)


## 5. Two or three documents, both ways


In [ ]:
# [CHANGE] test ids from the Day 1 40-document file
DOC_IDS = ["B01", "B12"]

import civicdesk_ft as ft

try:
    redact = ft.load_redactor()
    print("live redactor")
    from pathlib import Path
    import json, sys
    sys.path.insert(0, str(Path("../shared/resume_redaction").resolve()))
    import redaction_eval as R
    bench = R.load_benchmark()
    by = {d["id"]: d for d in bench["docs"]}
    for did in DOC_IDS:
        doc = by[did]
        print("=" * 60, did)
        for flag, tag in ((False, "base"), (True, "adapter")):
            out = redact(doc["text"], use_adapter=flag)
            sc = R.score_doc(doc, out)
            fired = R.release_checks(out, doc.get("form"))
            decision, reasons = R.route(fired, out, doc)
            print("---", tag, "---")
            print(out[:400])
            print("leaked", sc["leaked"], "route", decision)
except Exception as exc:
    print("live unavailable:", exc)
    ft.docs_from_runs(DOC_IDS)


## 6. Failure analysis

List three criteria cases the **base** (or the floor) still gets wrong. For each: more data, a rule, RAG, or a human?


In [ ]:
# [CHANGE] keep three ids after you read the printout
STILL_WRONG = []

import civicdesk_ft as ft

print("from this session:")
ft.still_wrong_criteria()
print("your list:", STILL_WRONG)


## 7. Probes — an adapter is not a database


In [ ]:
# [RUN]
import json
from pathlib import Path
import civicdesk_ft as ft

probe = json.loads(Path("data/demo/not_a_database_probe.json").read_text(encoding="utf-8"))
print(probe.get("question") or probe)
print("An adapter does not look up a handbook. That was Day 2.")


In [ ]:
# [OPTIONAL] tickets Track A preview (full treatment in T1-13)
import civicdesk_ft as ft

ft.show_failure("T1")


In [ ]:
# [OPTIONAL] unfamiliar layout / cabinet note (not in the redaction train set)
import json
from pathlib import Path

docs = json.loads(Path("data/demo/unfamiliar_layouts.json").read_text(encoding="utf-8"))
print(docs.get("note"))
for d in docs["docs"]:
    print("---", d["id"], d["layout"], "---")
    print(d["text"][:300])


In [ ]:
# [OPTIONAL] regex floor on one stress CV (emails and phones only)
import civicdesk_ft as ft

ft.print_redaction_floor("R2")


Phi-4 Mini is a generate-only column here unless a measured `phi_baseline_v1.json` exists.
